# Per-Day Brain-Region Scatter Decoding Orchestrator

Runs semantic category classification per **(patient, day, region, brain-resample)** unit subsamples.

- 5 brain resamples per region (subsampling 8 units) when region has >8 electrodes; 1 resample if exactly 8
- 10 semantic-category resamples per brain resample (run in a single SLURM job)
- Optionally borrows consensus hyperparams from the patient-level run (`REUSE_PATIENT_CONSENSUS = True`)
- Worker: `scat_classifier_region_worker.py`

### 1. Imports

In [1]:
import json
import subprocess
from pathlib import Path

import dill as pickle
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)


### 2. Configuration

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
PATIENTS    = ['YEY', 'YEZ', 'YFA', 'YFB', 'YFC', 'YFD', 'YFE', 'YFF', 'YFG',
               'YFI', 'YFK', 'YFL', 'YFM', 'YFN', 'YFP', 'YFQ', 'YFR', 'YFT', 'YFU', 'YFV']

VAD_ROOT    = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new')
LEGACY_ROOT = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_out')

BASE_RUN_NAME         = 'scat_xgboost_region_per_day'
PATIENT_CONSENSUS_RUN = 'scat_xgboost_sampled_norm_filtered'  # source of borrowed params

CLUSTER_PREDS_OVERRIDE = {}
FRS_PATH_OVERRIDE      = {}

# ── Day-partition settings ────────────────────────────────────────────────────
LOCAL_TZ           = 'America/Chicago'
ACTIVE_HOURS_START = 9
ACTIVE_HOURS_END   = 23
MIN_WORDS_PER_DAY  = 200

# ── Region settings ───────────────────────────────────────────────────────────
N_UNITS        = 8    # number of units to subsample per brain resample
N_BRAIN_RESAMPLES_FULL = 5   # when region has > N_UNITS electrodes
N_CAT_RESAMPLES        = 10  # category resamples per brain resample (run inside the job)

# ── Two-phase strategy ────────────────────────────────────────────────────────
# True  → borrow consensus params from the patient-level run (recommended)
# False → brain_resample=0 does grid search; remaining brain resamples use consensus
REUSE_PATIENT_CONSENSUS = False

SEED_STRIDE = 42

PYTHON_BIN  = '/scratch/tahaismail424/miniforge3/envs/speech_247_env/bin/python'
WORKER_PATH = Path('/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!'
                   '/standard_decoding_analysis/scat_classifier_region_worker.py')

PARTITION = 'Universal'
QOS       = 'big_batch_tier'
CPUS      = 8
MEM       = '40G'
TIME      = '48:00:00'

TEST_SIZE    = 0.2
SEARCH_ITERS = 40
CV_SPLITS    = 4
N_SHUFFLES   = 50
SCORING      = 'f1_macro'
PERM_METRIC  = 'f1_macro'

print('patients:', PATIENTS)
print('worker:', WORKER_PATH)
print(f'active hours: {ACTIVE_HOURS_START}:00 – {ACTIVE_HOURS_END}:00 {LOCAL_TZ}')
print(f'reuse patient consensus: {REUSE_PATIENT_CONSENSUS}')
print(f'n_units per brain resample: {N_UNITS}')
print(f'n_brain_resamples (when >8 units): {N_BRAIN_RESAMPLES_FULL}')
print(f'n_cat_resamples per brain resample: {N_CAT_RESAMPLES}')


patients: ['YEY', 'YEZ', 'YFA', 'YFB', 'YFC', 'YFD', 'YFE', 'YFF', 'YFG', 'YFI', 'YFK', 'YFL', 'YFM', 'YFN', 'YFP', 'YFQ', 'YFR', 'YFT', 'YFU', 'YFV']
worker: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/standard_decoding_analysis/scat_classifier_region_worker.py
active hours: 9:00 – 23:00 America/Chicago
reuse patient consensus: False
n_units per brain resample: 8
n_brain_resamples (when >8 units): 5
n_cat_resamples per brain resample: 10


### 3. Region Audit

Check which patients have `micro_info.csv` with region annotations, and how many units per region.

In [3]:
# Load micro_info for all patients that have the file
micro_dfs    = {}   # patient -> DataFrame
region_audit = []   # rows for display

for patient in PATIENTS:
    micro_path = VAD_ROOT / patient / f'{patient}_micro_info.csv'
    if not micro_path.exists():
        print(f'  [{patient}] no micro_info.csv — skipping')
        continue
    df = pd.read_csv(micro_path, index_col=0)
    micro_dfs[patient] = df
    for region, grp in df.groupby('region_symbol'):
        n = len(grp)
        n_brain = N_BRAIN_RESAMPLES_FULL if n > N_UNITS else 1
        region_audit.append({
            'patient': patient,
            'region': region,
            'n_units': n,
            'n_brain_resamples': n_brain,
            'eligible': n >= N_UNITS,
        })

audit_df = pd.DataFrame(region_audit)
print(f'Patients with micro_info.csv: {len(micro_dfs)}')
print(f'Patients without: {sorted(set(PATIENTS) - set(micro_dfs))}')
print(f'Total eligible (patient, region) pairs: {int(audit_df["eligible"].sum())}')
print()
display(audit_df)


Patients with micro_info.csv: 20
Patients without: []
Total eligible (patient, region) pairs: 61



,patient,region,n_units,n_brain_resamples,eligible
0,YEY,HPC,16,5,True
1,YEZ,HPC,16,5,True
2,YEZ,PCC,8,1,True
3,YFA,ACC,8,1,True
4,YFA,HPC,32,5,True
5,YFB,ACC,8,1,True
6,YFB,HPC,32,5,True
7,YFC,ACC,32,5,True
8,YFC,HPC,32,5,True
9,YFD,ACC,16,5,True


### 4. Compute Per-Day Word Indices Per Patient

In [4]:
def compute_day_indices(patient: str) -> 'dict[str, Path]':
    patient_root = VAD_ROOT / patient
    filtered_csv = patient_root / f'{patient}_word_df_filtered.csv'

    if not filtered_csv.exists():
        print(f'  [{patient}] missing {filtered_csv.name}')
        return {}

    df = pd.read_csv(filtered_csv, usecols=['original_word_idx', 'utc_word_start'])
    df['utc_word_start'] = pd.to_datetime(df['utc_word_start'], utc=True)
    df['local_dt'] = df['utc_word_start'].dt.tz_convert(LOCAL_TZ)
    df['hour']     = df['local_dt'].dt.hour
    df['date_str'] = df['local_dt'].dt.strftime('%Y-%m-%d')

    active = df[(df['hour'] >= ACTIVE_HOURS_START) & (df['hour'] < ACTIVE_HOURS_END)]

    day_paths = {}
    for date_str, grp in active.groupby('date_str'):
        if len(grp) < MIN_WORDS_PER_DAY:
            print(f'  [{patient}] {date_str}: only {len(grp)} words — skipping')
            continue
        idx_dir  = patient_root / 'standard_decoding_analysis' / BASE_RUN_NAME / date_str
        idx_dir.mkdir(parents=True, exist_ok=True)
        idx_path = idx_dir / f'{patient}_{date_str}_word_idx.npy'
        word_idx = grp['original_word_idx'].values.astype(np.int64)
        np.save(idx_path, word_idx)
        day_paths[date_str] = idx_path
        print(f'  [{patient}] {date_str}: {len(word_idx):,} words → {idx_path.name}')

    return day_paths


print(f'Computing per-day word indices for {len(micro_dfs)} patients with region info...\n')
patient_day_idx_paths = {}
for patient in micro_dfs:
    day_paths = compute_day_indices(patient)
    if day_paths:
        patient_day_idx_paths[patient] = day_paths

total_pairs = sum(len(v) for v in patient_day_idx_paths.values())
print(f'\nTotal (patient, day) pairs ready: {total_pairs}')


Computing per-day word indices for 20 patients with region info...

  [YEY] 2024-04-01: 16,510 words → YEY_2024-04-01_word_idx.npy
  [YEY] 2024-04-02: 23,841 words → YEY_2024-04-02_word_idx.npy
  [YEZ] 2024-04-09: 6,698 words → YEZ_2024-04-09_word_idx.npy
  [YEZ] 2024-04-10: 16,464 words → YEZ_2024-04-10_word_idx.npy
  [YEZ] 2024-04-11: 1,584 words → YEZ_2024-04-11_word_idx.npy
  [YEZ] 2024-04-12: 7,996 words → YEZ_2024-04-12_word_idx.npy
  [YEZ] 2024-04-14: 18,298 words → YEZ_2024-04-14_word_idx.npy
  [YEZ] 2024-04-15: 14,595 words → YEZ_2024-04-15_word_idx.npy
  [YFA] 2024-04-23: 18,729 words → YFA_2024-04-23_word_idx.npy
  [YFA] 2024-04-24: 58,808 words → YFA_2024-04-24_word_idx.npy
  [YFA] 2024-04-25: 48,473 words → YFA_2024-04-25_word_idx.npy
  [YFA] 2024-04-26: 36,215 words → YFA_2024-04-26_word_idx.npy
  [YFA] 2024-04-27: 66,950 words → YFA_2024-04-27_word_idx.npy
  [YFA] 2024-04-28: 43,939 words → YFA_2024-04-28_word_idx.npy
  [YFA] 2024-04-29: 54,695 words → YFA_2024-04-29_wor

### 5. Path Resolution & Helpers

In [5]:
def first_existing(paths):
    for p in paths:
        if p is not None and Path(p).exists():
            return Path(p)
    return None


def resolve_patient_neural_inputs(patient: str) -> dict:
    root = VAD_ROOT / patient
    cluster_override = CLUSTER_PREDS_OVERRIDE.get(patient)
    frs_override     = FRS_PATH_OVERRIDE.get(patient)

    new_clusters = first_existing([root / 'embeddings' / 'semantic_cluster_predictions.npy'])
    new_frs = first_existing([
        root / 'neural_embeddings' / 'word_frs.npy',
        root / 'all_convo_recording' / 'word_avg_frs.npy',
    ])
    legacy_frs = first_existing([LEGACY_ROOT / patient / 'all_convo_recording' / 'word_avg_frs.npy'])

    if cluster_override is not None or frs_override is not None:
        cluster_preds_path = first_existing([cluster_override, new_clusters])
        frs_path           = first_existing([frs_override, new_frs, legacy_frs])
    elif new_clusters is not None and new_frs is not None:
        cluster_preds_path = new_clusters
        frs_path           = new_frs
    else:
        cluster_preds_path = None
        frs_path           = None

    return {'cluster_preds_path': cluster_preds_path, 'frs_path': frs_path}


def patient_consensus_path(patient: str) -> 'Path | None':
    p = VAD_ROOT / patient / 'standard_decoding_analysis' / PATIENT_CONSENSUS_RUN / 'consensus_best_params.json'
    return p if p.exists() else None


def choose_consensus_params(param_options: list) -> dict:
    params = dict(param_options[0])
    params['max_depth']        = int(np.min([p['max_depth'] for p in param_options]))
    params['min_child_weight'] = int(np.max([p['min_child_weight'] for p in param_options]))
    params['gamma']            = float(np.max([p['gamma'] for p in param_options]))
    params['reg_lambda']       = float(np.max([p['reg_lambda'] for p in param_options]))
    params['reg_alpha']        = float(np.max([p['reg_alpha'] for p in param_options]))
    params['subsample']        = float(np.median([p['subsample'] for p in param_options]))
    params['colsample_bytree'] = float(np.median([p['colsample_bytree'] for p in param_options]))
    params['learning_rate']    = float(np.median([p['learning_rate'] for p in param_options]))
    params['n_estimators']     = int(np.median([p['n_estimators'] for p in param_options]))
    params['tree_method'] = 'hist'
    params['device'] = 'cpu'
    params.pop('predictor', None)
    params.pop('use_label_encoder', None)
    return params


### 6. Build Status Table

In [6]:
def brain_resample_paths(out_root: Path, region: str, b: int, n_cat: int) -> dict:
    """Paths for one (region, brain_resample) job."""
    return {
        'success':      out_root / f'region_{region}_brain{b}_SUCCESS',
        'error':        out_root / f'region_{region}_brain{b}_error.txt',
        'best_params':  out_root / f'best_params_{region}_brain{b}.json',
        'cat_pkls':     [out_root / f'scat_region_{region}_brain{b}_cat{c}.pkl' for c in range(n_cat)],
        'cat_summaries':[out_root / f'summary_{region}_brain{b}_cat{c}.json'    for c in range(n_cat)],
    }


def build_status_df() -> pd.DataFrame:
    rows = []
    for patient, day_idx_map in patient_day_idx_paths.items():
        neural = resolve_patient_neural_inputs(patient)
        has_inputs = (neural['cluster_preds_path'] is not None
                      and neural['frs_path'] is not None)
        borrowed_consensus = patient_consensus_path(patient) if REUSE_PATIENT_CONSENSUS else None

        micro_df = micro_dfs[patient]

        for date_str, word_idx_path in day_idx_map.items():
            day_base = VAD_ROOT / patient / 'standard_decoding_analysis' / BASE_RUN_NAME / date_str
            day_base.mkdir(parents=True, exist_ok=True)

            for region, region_grp in micro_df.groupby('region_symbol'):
                n_units = len(region_grp)
                if n_units < N_UNITS:
                    continue  # not enough units to subsample

                n_brain = N_BRAIN_RESAMPLES_FULL if n_units > N_UNITS else 1
                region_base = day_base / region
                region_base.mkdir(parents=True, exist_ok=True)
                micro_info_path = VAD_ROOT / patient / f'{patient}_micro_info.csv'

                consensus_json = region_base / 'consensus_best_params.json'

                # Copy patient-level consensus into region dir if reusing
                if REUSE_PATIENT_CONSENSUS and borrowed_consensus is not None and not consensus_json.exists():
                    consensus_json.write_text(borrowed_consensus.read_text())

                consensus_ready = consensus_json.exists()

                if REUSE_PATIENT_CONSENSUS:
                    phase1_brains = []
                    phase2_brains = list(range(n_brain))
                else:
                    phase1_brains = [0]             # brain_resample=0 does grid search
                    phase2_brains = list(range(1, n_brain))

                brain0_done = (
                    (region_base / f'brain0' / f'region_{region}_brain0_SUCCESS').exists()
                    if not REUSE_PATIENT_CONSENSUS else True
                )

                for b in range(n_brain):
                    out_root = region_base / f'brain{b}'
                    out_root.mkdir(parents=True, exist_ok=True)
                    paths = brain_resample_paths(out_root, region, b, N_CAT_RESAMPLES)

                    cat_done    = [p.exists() for p in paths['cat_pkls']]
                    cat_summary = [p.exists() for p in paths['cat_summaries']]

                    rows.append({
                        'patient':          patient,
                        'date':             date_str,
                        'region':           region,
                        'brain_resample':   b,
                        'n_region_units':   n_units,
                        'n_brain_resamples':n_brain,
                        'stage':            'hyperparam' if b in phase1_brains else 'fixed',
                        'cluster_preds_path': neural['cluster_preds_path'],
                        'frs_path':           neural['frs_path'],
                        'micro_info_path':    micro_info_path,
                        'word_idx_path':      word_idx_path,
                        'has_inputs':         has_inputs,
                        'out_root':           out_root,
                        'region_base':        region_base,
                        'consensus_json':     consensus_json,
                        'consensus_ready':    consensus_ready,
                        'brain0_done':        brain0_done,
                        'has_success':        paths['success'].exists(),
                        'has_error':          paths['error'].exists(),
                        'n_cat_done':         sum(cat_done),
                        'n_cat_summary':      sum(cat_summary),
                        'success_path':       paths['success'],
                        'error_path':         paths['error'],
                        'best_params_path':   paths['best_params'],
                        'ready_phase1':       (
                            has_inputs
                            and b in phase1_brains
                            and not paths['success'].exists()
                        ),
                        'ready_phase2':       (
                            has_inputs
                            and b in phase2_brains
                            and brain0_done
                            and consensus_ready
                            and not paths['success'].exists()
                        ),
                    })
    return pd.DataFrame(rows).sort_values(
        ['patient', 'date', 'region', 'brain_resample']
    ).reset_index(drop=True)


status_df = build_status_df()
print(f'Total rows (patient × day × region × brain_resample): {len(status_df)}')
print(f'has inputs:      {int(status_df["has_inputs"].sum())}')
print(f'consensus ready: {int(status_df["consensus_ready"].sum())}')
print(f'completed:       {int(status_df["has_success"].sum())}')
print(f'errors:          {int(status_df["has_error"].sum())}')
print(f'ready_phase1:    {int(status_df["ready_phase1"].sum())}')
print(f'ready_phase2:    {int(status_df["ready_phase2"].sum())}')
status_df[['patient','date','region','brain_resample','n_region_units',
           'stage','has_inputs','consensus_ready','has_success','has_error',
           'ready_phase1','ready_phase2']].head(40)


Total rows (patient × day × region × brain_resample): 1515
has inputs:      1515
consensus ready: 1406
completed:       1424
errors:          15
ready_phase1:    15
ready_phase2:    0


,patient,date,region,brain_resample,n_region_units,stage,has_inputs,consensus_ready,has_success,has_error,ready_phase1,ready_phase2
0,YEY,2024-04-01,HPC,0,16,hyperparam,True,True,True,False,False,False
1,YEY,2024-04-01,HPC,1,16,fixed,True,True,True,False,False,False
2,YEY,2024-04-01,HPC,2,16,fixed,True,True,True,False,False,False
3,YEY,2024-04-01,HPC,3,16,fixed,True,True,True,False,False,False
4,YEY,2024-04-01,HPC,4,16,fixed,True,True,True,False,False,False
5,YEY,2024-04-02,HPC,0,16,hyperparam,True,True,True,False,False,False
6,YEY,2024-04-02,HPC,1,16,fixed,True,True,True,False,False,False
7,YEY,2024-04-02,HPC,2,16,fixed,True,True,True,False,False,False
8,YEY,2024-04-02,HPC,3,16,fixed,True,True,True,False,False,False
9,YEY,2024-04-02,HPC,4,16,fixed,True,True,True,False,False,False


### 7. Patient × Day × Region Summary

In [7]:
pair_df = (
    status_df.drop_duplicates(subset=['patient', 'date', 'region'])
    [['patient', 'date', 'region', 'n_region_units', 'n_brain_resamples',
      'has_inputs', 'consensus_ready']]
    .reset_index(drop=True)
)
print(f'Unique (patient, day, region) pairs: {len(pair_df)}')
display(pair_df)


Unique (patient, day, region) pairs: 447


,patient,date,region,n_region_units,n_brain_resamples,has_inputs,consensus_ready
0,YEY,2024-04-01,HPC,16,5,True,True
1,YEY,2024-04-02,HPC,16,5,True,True
2,YEZ,2024-04-09,HPC,16,5,True,True
3,YEZ,2024-04-09,PCC,8,1,True,True
4,YEZ,2024-04-10,HPC,16,5,True,True
...,...,...,...,...,...,...,...
442,YFV,2026-02-20,A,16,5,True,True
443,YFV,2026-02-20,ACC,16,5,True,True
444,YFV,2026-02-20,HPC,32,5,True,True
445,YFV,2026-02-20,OFC,16,5,True,True


### 8a. Submit Phase-1 Jobs

*(Only runs if `REUSE_PATIENT_CONSENSUS = False`; brain_resample=0 does grid search.)*

In [8]:
DRY_RUN = False

submitted = []
failed    = []

for _, row in status_df[status_df['ready_phase1']].iterrows():
    patient = row['patient']
    date_str = row['date']
    region   = row['region']
    b        = int(row['brain_resample'])

    logs_dir    = row['out_root'] / 'slurm_logs'
    scripts_dir = row['out_root'] / 'slurm_scripts'
    logs_dir.mkdir(parents=True, exist_ok=True)
    scripts_dir.mkdir(parents=True, exist_ok=True)

    word_idx_arg = f'--word-idx-path "{row["word_idx_path"]}"' if row['word_idx_path'] else ''
    job_name = f'rgn_p1_{patient}_{region}_{date_str}_b{b}'
    sbatch_text = f"""#!/bin/bash
#SBATCH --job-name={job_name}
#SBATCH --partition={PARTITION}
#SBATCH --qos={QOS}
#SBATCH --cpus-per-task={CPUS}
#SBATCH --mem={MEM}
#SBATCH --time={TIME}
#SBATCH --output={logs_dir}/{patient}_{region}_b{b}_%j.out
#SBATCH --error={logs_dir}/{patient}_{region}_b{b}_%j.err

set -eo pipefail
echo "HOSTNAME: $(hostname)"
echo "START: $(date)"
echo "PATIENT: {patient}  REGION: {region}  DATE: {date_str}  BRAIN: {b}"

{PYTHON_BIN} {WORKER_PATH} \\
  --patient {patient} \\
  --region {region} \\
  --brain-resample-idx {b} \\
  --micro-info-path "{row['micro_info_path']}" \\
  --frs-path "{row['frs_path']}" \\
  --cluster-preds-path "{row['cluster_preds_path']}" \\
  --out-dir "{row['out_root']}" \\
  {word_idx_arg} \\
  --n-units {N_UNITS} \\
  --n-cat-resamples {N_CAT_RESAMPLES} \\
  --seed-stride {SEED_STRIDE} \\
  --test-size {TEST_SIZE} \\
  --n-iter {SEARCH_ITERS} \\
  --cv-splits {CV_SPLITS} \\
  --n-shuffles {N_SHUFFLES} \\
  --n-jobs {CPUS * 2} \\
  --scoring {SCORING} \\
  --perm-metric {PERM_METRIC}

echo "END: $(date)"
"""
    sbatch_path = scripts_dir / f'{patient}_{region}_brain{b}.sbatch'
    sbatch_path.write_text(sbatch_text)

    if DRY_RUN:
        submitted.append((patient, date_str, region, b, 'dry-run'))
        print(f'dry-run: {patient} {date_str} {region} brain={b}')
    else:
        try:
            res = subprocess.run(['sbatch', str(sbatch_path)], capture_output=True, text=True, check=True)
            submitted.append((patient, date_str, region, b, res.stdout.strip()))
            print(f'submitted: {patient} {date_str} {region} brain={b} -> {res.stdout.strip()}')
        except subprocess.CalledProcessError as exc:
            failed.append((patient, date_str, region, b, exc.stderr.strip()))
            print(f'FAILED: {patient} {date_str} {region} brain={b}\n{exc.stderr}')

print(f'phase-1 submitted={len(submitted)}, failed={len(failed)}')


submitted: YEZ 2024-04-11 HPC brain=0 -> Submitted batch job 477527
submitted: YEZ 2024-04-11 PCC brain=0 -> Submitted batch job 477528
submitted: YFG 2024-09-23 ACC brain=0 -> Submitted batch job 477529
submitted: YFG 2024-09-23 HPC brain=0 -> Submitted batch job 477530
submitted: YFG 2024-09-23 OFC brain=0 -> Submitted batch job 477531
submitted: YFG 2024-09-23 PCC brain=0 -> Submitted batch job 477532
submitted: YFM 2025-03-10 HPC brain=0 -> Submitted batch job 477533
submitted: YFM 2025-03-10 OFC brain=0 -> Submitted batch job 477534
submitted: YFM 2025-03-10 Th brain=0 -> Submitted batch job 477535
submitted: YFR 2025-06-30 A brain=0 -> Submitted batch job 477536
submitted: YFR 2025-06-30 ACC brain=0 -> Submitted batch job 477537
submitted: YFR 2025-06-30 HPC brain=0 -> Submitted batch job 477538
submitted: YFU 2025-12-08 A brain=0 -> Submitted batch job 477539
submitted: YFU 2025-12-08 ACC brain=0 -> Submitted batch job 477540
submitted: YFU 2025-12-08 HPC brain=0 -> Submitted ba

### 8b. Build Per-Day-Region Consensus Params

*(Only needed if `REUSE_PATIENT_CONSENSUS = False`; runs after brain_resample=0 finishes.)*

In [9]:
if not REUSE_PATIENT_CONSENSUS:
    built   = []
    blocked = []
    for (patient, date_str, region), grp in status_df.groupby(['patient', 'date', 'region']):
        region_base    = grp.iloc[0]['region_base']
        consensus_json = grp.iloc[0]['consensus_json']
        if consensus_json.exists():
            print(f'consensus already exists: {patient} {date_str} {region}')
            continue
        brain0_row = grp[grp['brain_resample'] == 0].iloc[0]
        if not brain0_row['has_success']:
            blocked.append((patient, date_str, region))
            continue
        bp_path = brain0_row['best_params_path']
        if not bp_path.exists():
            blocked.append((patient, date_str, region))
            continue
        params = json.loads(bp_path.read_text())
        consensus_json.write_text(json.dumps(params, indent=2))
        built.append((patient, date_str, region))
        print(f'built consensus: {patient} {date_str} {region}')
    print('built:', len(built), '  blocked:', len(blocked))
else:
    print('REUSE_PATIENT_CONSENSUS=True — consensus already copied during status build.')


consensus already exists: YEY 2024-04-01 HPC
consensus already exists: YEY 2024-04-02 HPC
consensus already exists: YEZ 2024-04-09 HPC
consensus already exists: YEZ 2024-04-09 PCC
consensus already exists: YEZ 2024-04-10 HPC
consensus already exists: YEZ 2024-04-10 PCC
consensus already exists: YEZ 2024-04-12 HPC
consensus already exists: YEZ 2024-04-12 PCC
consensus already exists: YEZ 2024-04-14 HPC
consensus already exists: YEZ 2024-04-14 PCC
consensus already exists: YEZ 2024-04-15 HPC
consensus already exists: YEZ 2024-04-15 PCC
consensus already exists: YFA 2024-04-23 ACC
consensus already exists: YFA 2024-04-23 HPC
consensus already exists: YFA 2024-04-24 ACC
consensus already exists: YFA 2024-04-24 HPC
consensus already exists: YFA 2024-04-25 ACC
consensus already exists: YFA 2024-04-25 HPC
consensus already exists: YFA 2024-04-26 ACC
consensus already exists: YFA 2024-04-26 HPC
consensus already exists: YFA 2024-04-27 ACC
consensus already exists: YFA 2024-04-27 HPC
consensus 

### 8c. Submit Phase-2 (Fixed-Params) Jobs

In [10]:
DRY_RUN = False

# Refresh status after any consensus files were written
status_df = build_status_df()

submitted = []
failed    = []

for _, row in status_df[status_df['ready_phase2']].iterrows():
    patient  = row['patient']
    date_str = row['date']
    region   = row['region']
    b        = int(row['brain_resample'])

    logs_dir    = row['out_root'] / 'slurm_logs'
    scripts_dir = row['out_root'] / 'slurm_scripts'
    logs_dir.mkdir(parents=True, exist_ok=True)
    scripts_dir.mkdir(parents=True, exist_ok=True)

    word_idx_arg = f'--word-idx-path "{row["word_idx_path"]}"' if row['word_idx_path'] else ''
    job_name = f'rgn_p2_{patient}_{region}_{date_str}_b{b}'
    sbatch_text = f"""#!/bin/bash
#SBATCH --job-name={job_name}
#SBATCH --partition={PARTITION}
#SBATCH --qos={QOS}
#SBATCH --cpus-per-task={CPUS}
#SBATCH --mem={MEM}
#SBATCH --time={TIME}
#SBATCH --output={logs_dir}/{patient}_{region}_b{b}_%j.out
#SBATCH --error={logs_dir}/{patient}_{region}_b{b}_%j.err

set -eo pipefail
echo "HOSTNAME: $(hostname)"
echo "START: $(date)"
echo "PATIENT: {patient}  REGION: {region}  DATE: {date_str}  BRAIN: {b}"

{PYTHON_BIN} {WORKER_PATH} \\
  --patient {patient} \\
  --region {region} \\
  --brain-resample-idx {b} \\
  --micro-info-path "{row['micro_info_path']}" \\
  --frs-path "{row['frs_path']}" \\
  --cluster-preds-path "{row['cluster_preds_path']}" \\
  --out-dir "{row['out_root']}" \\
  --params-json "{row['consensus_json']}" \\
  {word_idx_arg} \\
  --n-units {N_UNITS} \\
  --n-cat-resamples {N_CAT_RESAMPLES} \\
  --seed-stride {SEED_STRIDE} \\
  --test-size {TEST_SIZE} \\
  --n-iter {SEARCH_ITERS} \\
  --cv-splits {CV_SPLITS} \\
  --n-shuffles {N_SHUFFLES} \\
  --n-jobs {CPUS * 2} \\
  --scoring {SCORING} \\
  --perm-metric {PERM_METRIC}

echo "END: $(date)"
"""
    sbatch_path = scripts_dir / f'{patient}_{region}_brain{b}.sbatch'
    sbatch_path.write_text(sbatch_text)

    if DRY_RUN:
        submitted.append((patient, date_str, region, b, 'dry-run'))
        print(f'dry-run: {patient} {date_str} {region} brain={b}')
    else:
        try:
            res = subprocess.run(['sbatch', str(sbatch_path)], capture_output=True, text=True, check=True)
            submitted.append((patient, date_str, region, b, res.stdout.strip()))
            print(f'submitted: {patient} {date_str} {region} brain={b} -> {res.stdout.strip()}')
        except subprocess.CalledProcessError as exc:
            failed.append((patient, date_str, region, b, exc.stderr.strip()))
            print(f'FAILED: {patient} {date_str} {region} brain={b}\n{exc.stderr}')

print(f'phase-2 submitted={len(submitted)}, failed={len(failed)}')


submitted: YFB 2024-05-02 HPC brain=1 -> Submitted batch job 477542
submitted: YFB 2024-05-02 HPC brain=2 -> Submitted batch job 477543
submitted: YFB 2024-05-02 HPC brain=3 -> Submitted batch job 477544
submitted: YFB 2024-05-02 HPC brain=4 -> Submitted batch job 477545
submitted: YFB 2024-05-03 HPC brain=1 -> Submitted batch job 477546
submitted: YFB 2024-05-03 HPC brain=2 -> Submitted batch job 477547
submitted: YFB 2024-05-03 HPC brain=3 -> Submitted batch job 477548
submitted: YFB 2024-05-03 HPC brain=4 -> Submitted batch job 477549
submitted: YFB 2024-05-04 HPC brain=1 -> Submitted batch job 477550
submitted: YFB 2024-05-04 HPC brain=2 -> Submitted batch job 477551
submitted: YFB 2024-05-04 HPC brain=3 -> Submitted batch job 477552
submitted: YFB 2024-05-04 HPC brain=4 -> Submitted batch job 477553
submitted: YFB 2024-05-05 HPC brain=1 -> Submitted batch job 477554
submitted: YFB 2024-05-05 HPC brain=2 -> Submitted batch job 477555
submitted: YFB 2024-05-05 HPC brain=3 -> Submitt

### 9. Check Status

In [ ]:
status_df = build_status_df()
pair_progress = (
    status_df.groupby(['patient', 'date', 'region'])
    .agg(
        total_brain=('brain_resample', 'count'),
        done=('has_success', 'sum'),
        errors=('has_error', 'sum'),
        cat_done=('n_cat_done', 'sum'),
    )
    .reset_index()
)
pair_progress['total_cat'] = pair_progress['total_brain'] * N_CAT_RESAMPLES
pair_progress['pct_brain'] = (pair_progress['done'] / pair_progress['total_brain'] * 100).round(0).astype(int)
pair_progress['pct_cat']   = (pair_progress['cat_done'] / pair_progress['total_cat'] * 100).round(0).astype(int)
print(f'Overall: {int(status_df["has_success"].sum())} / {len(status_df)} brain resamples done')
print(f'Cat resamples: {int(status_df["n_cat_done"].sum())} / {len(status_df) * N_CAT_RESAMPLES}')
display(pair_progress)


### 10. Inspect Errors

In [ ]:
error_rows = status_df[status_df['has_error']].copy()
print(f'Brain resamples with error files: {len(error_rows)}')
for _, row in error_rows.head(10).iterrows():
    print('=' * 100)
    print(row['patient'], row['date'], row['region'], f'brain={row["brain_resample"]}')
    print(Path(row['error_path']).read_text()[:4000])


### 11. Aggregate Results Per (Patient, Day, Region)

In [ ]:
status_df = build_status_df()

for (patient, date_str, region), grp in status_df.groupby(['patient', 'date', 'region']):
    if not bool(grp['has_success'].all()):
        n_done = int(grp['has_success'].sum())
        print(f'skipping {patient} {date_str} {region}: {n_done}/{len(grp)} brain resamples done')
        continue

    region_base  = grp.iloc[0]['region_base']
    agg_csv      = region_base / 'scat_region_all_resamples.csv'
    agg_pkl      = region_base / 'scat_region_all_resamples.pkl'

    summary_rows = []
    aggregate    = {}

    n_brain = int(grp['n_brain_resamples'].iloc[0])
    for b in range(n_brain):
        out_root = region_base / f'brain{b}'
        for c in range(N_CAT_RESAMPLES):
            pkl_path  = out_root / f'scat_region_{region}_brain{b}_cat{c}.pkl'
            summ_path = out_root / f'summary_{region}_brain{b}_cat{c}.json'
            if not pkl_path.exists() or not summ_path.exists():
                continue
            with open(summ_path) as f:
                summary_rows.append(json.load(f))
            with open(pkl_path, 'rb') as f:
                aggregate[f'brain{b}_cat{c}'] = pickle.load(f)

    if not summary_rows:
        print(f'no completed cat resamples for {patient} {date_str} {region}')
        continue

    summary_df = pd.DataFrame(summary_rows).sort_values(
        ['brain_resample_idx', 'cat_resample_idx']
    ).reset_index(drop=True)
    summary_df.to_csv(agg_csv, index=False)
    with open(agg_pkl, 'wb') as f:
        pickle.dump(aggregate, f)
    print(f'aggregated {patient} {date_str} {region} ({len(summary_rows)} cat resamples) -> {agg_csv}')
    display(summary_df[['brain_resample_idx','cat_resample_idx','acc','balanced_accuracy',
                          'f1_macro','perm_pvalue','n_region_units','n_units_used']].head(15))
